# Tutorial Interactivo statscepal: CEPALSTAT, Bitácoras y Visualización

Este notebook demuestra el flujo de trabajo completo del ecosistema `statscepal`:
1. **Búsqueda Instantánea en el Catálogo Maestro (2,070 indicadores)** sin llamadas de red.
2. **Consulta a Bitácoras Certificadas** para ahorrar un 85% de tokens y evitar sesgos metodológicos.
3. **Extracción y Visualización** de series de tiempo con `pandas`, `matplotlib` y `seaborn`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statscepal import CepalStatClient, DataExtractor, DimensionManager
from statscepal.catalog import CatalogStore
from statscepal.bitacora import BitacoraStore, AntiBiasValidator

sns.set_theme(style="whitegrid")

## 1. Búsqueda Offline en el Catálogo Maestro (100% Local)

In [ ]:
catalog = CatalogStore()
stats = catalog.get_stats()
print(f"Total de indicadores indexados en SQLite: {stats.total_indicators}")
for theme, count in stats.by_theme.items():
    print(f"  - {theme}: {count}")

# Búsqueda instantánea
resultados = catalog.search("poblacion", limit=5)
for r in resultados:
    print(f"[{r.indicator_id}] {r.name} ({r.theme})")

## 2. Consulta a la Bitácora Certificada (Chile: Población Total)

Leemos la receta certificada para conocer la unidad oficial y las alertas de sesgo (año base de corte 2021 de CELADE).

In [ ]:
store = BitacoraStore()
recetas = store.lookup("poblacion", country="Chile")

receta_chile = recetas[0]
print("ID de Receta:", receta_chile.id)
print("Indicador:", receta_chile.indicator_name)
print("Unidad:", receta_chile.unit)
print("Alerta de Escala:", receta_chile.antibias.scale_factor_note)
print("¿Es proyección post-2021?:", receta_chile.antibias.is_projection)
print("Advertencias:", receta_chile.antibias.warnings)

## 3. Extracción de Datos y Gráfico de Tendencia Histórica vs. Proyección

In [ ]:
with CepalStatClient() as client:
    extractor = DataExtractor(client)
    # Extraer para Chile (224) para todos los años disponibles
    df = extractor.get_dataframe(4788, members=[224, 88623], decode_labels=True)

# Formatear columnas
df["Año"] = pd.to_numeric(df["Años__ESTANDAR"], errors="coerce")
df = df.sort_values("Año").reset_index(drop=True)

# Gráfico separando datos históricos y proyecciones CELADE (corte 2021)
plt.figure(figsize=(10, 5))
historico = df[df["Año"] <= 2021]
proyeccion = df[df["Año"] >= 2021]

plt.plot(historico["Año"], historico["value"] / 1000, label="Estimación Observada (Censal)", color="#1f77b4", lw=2.5)
plt.plot(proyeccion["Año"], proyeccion["value"] / 1000, label="Proyección CELADE (post-2021)", color="#ff7f0e", lw=2.5, ls="--")
plt.axvline(2021, color="gray", linestyle=":", label="Año Base de Proyección (2021)")

plt.title("Chile: Evolución y Proyección de la Población Total (1950 - 2100)", fontsize=14, fontweight="bold")
plt.xlabel("Año", fontsize=12)
plt.ylabel("Millones de Personas", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()